In [59]:
import os
import json
from pprint import pprint
from typing import Any, Dict, List
import re
import html as htmllib
from bs4 import BeautifulSoup

In [60]:
taint_result_root_directory = r"C:\Users\ASUS\anaconda3-project-code\wearable-2-appshark\appshark-output\app-standalone"

In [61]:
# Function get all sub-directory under root directory
def get_all_subdirectories_under_one_level(root_dir: str):
    """
    Get all subdirectories directly under the given root directory (one level only).
    Return a list of full paths.
    """
    return [
        os.path.join(root_dir, name)
        if not os.path.isabs(name) else name
        for name in os.listdir(root_dir)
        if os.path.isdir(os.path.join(root_dir, name))
    ]
def get_app_package_name(path: str) -> str:
    """
    Return the last directory name from a full path.
    Example:
        E:\\wearable-2-dataset-code-extraction\\adarshurs.android.vlcmobileremote
        -> 'adarshurs.android.vlcmobileremote'
    """
    return os.path.basename(os.path.normpath(path))
def get_html_files_in_directory(directory_path: str):
    """
    Trả về danh sách (list) chứa đường dẫn đầy đủ của tất cả file .html
    trong thư mục được chỉ định (không đệ quy).
    """
    html_files = []

    # Kiểm tra thư mục tồn tại
    if not os.path.isdir(directory_path):
        raise FileNotFoundError(f"Thư mục không tồn tại: {directory_path}")

    # Duyệt qua tất cả file trong thư mục
    for filename in os.listdir(directory_path):
        if filename.lower().endswith(".html"):
            full_path = os.path.join(directory_path, filename)
            html_files.append(full_path)

    return html_files
def extract_java_blocks_from_html(input_html_path: str, output_dir: str):
    """
    Trích xuất các khối code Java từ file HTML và lưu thành:
      <html_basename>-code-block-<i>.java

    VD:
      input_html_path = .../4-PersonalInfo_Network.html
      -> 4-PersonalInfo_Network-code-block-1.java, 4-PersonalInfo_Network-code-block-2.java, ...

    Returns
    -------
    list[str]: Danh sách đường dẫn các file .java đã lưu.
    """
    # Tạo thư mục đích nếu chưa có
    os.makedirs(output_dir, exist_ok=True)

    # Lấy base name từ file HTML và làm sạch cho an toàn tên file
    html_basename = os.path.splitext(os.path.basename(input_html_path))[0]
    html_basename = re.sub(r'[^A-Za-z0-9._\-]+', '_', html_basename)  # giữ chữ/số/._-

    # Đọc nội dung HTML
    with open(input_html_path, "r", encoding="utf-8") as f:
        html_content = f.read()

    # Parse HTML
    soup = BeautifulSoup(html_content, "html.parser")

    # Tìm các code block Java
    code_nodes = soup.select("code.java, code.language-java, code[class*=java]")

    saved_files = []
    for x, node in enumerate(code_nodes, start=1):
        # Lấy text & unescape
        java_text = node.get_text("\n")
        java_text = htmllib.unescape(java_text).strip() + "\n"

        # Đặt tên theo yêu cầu
        filename = f"{html_basename}-code-block-{x}.java"
        output_path = os.path.join(output_dir, filename)

        # Ghi file
        with open(output_path, "w", encoding="utf-8") as out:
            out.write(java_text)

        saved_files.append(output_path)

    # print(f"✅ Extracted {len(saved_files)} Java code block(s) from '{input_html_path}'")
    # for fpath in saved_files:
    #     print(" -", fpath)

    return saved_files
def cleanup_non_java_files(java_dir: str):
    """
    Duyệt qua toàn bộ file .java trong thư mục chỉ định.
    Xóa những file KHÔNG phải Java source code (ví dụ Jimple/IR, file rỗng, log, ...).

    Parameters
    ----------
    java_dir : str
        Đường dẫn thư mục chứa các file .java

    Returns
    -------
    dict
        {
            "checked": int,    # tổng số file .java đã kiểm tra
            "deleted": list    # danh sách file bị xóa
        }
    """

    # --- Kiểm tra thư mục tồn tại ---
    if not os.path.exists(java_dir):
        print(f"⚠️ Thư mục '{java_dir}' không tồn tại — bỏ qua kiểm tra.")
        return {"checked": 0, "deleted": []}

    if not os.path.isdir(java_dir):
        print(f"⚠️ '{java_dir}' không phải là thư mục — bỏ qua kiểm tra.")
        return {"checked": 0, "deleted": []}

    # Lấy danh sách file .java trong thư mục
    java_files = [f for f in os.listdir(java_dir) if f.endswith(".java")]
    if not java_files:
        print(f"ℹ️ Không tìm thấy file .java trong thư mục '{java_dir}'.")
        return {"checked": 0, "deleted": []}

    # --- Mẫu nhận diện Java thực ---
    JAVA_KEYWORDS = {
        "class","interface","enum","package","import",
        "public","private","protected","static","final","void","new","return",
        "extends","implements","throws","try","catch"
    }
    JAVA_METHOD_SIG = re.compile(
        r'\b(public|private|protected|static|final)\b.*\b[A-Za-z_]\w*\s*\([^)]*\)\s*\{'
    )
    JAVA_CLASS_DECL = re.compile(r'\b(class|interface|enum)\s+[A-Za-z_]\w*')

    # --- Mẫu đặc trưng của IR/Jimple/AppShark (cần loại bỏ) ---
    NON_JAVA_IR_PATTERNS = [
        re.compile(r'\b:=\b'),
        re.compile(r'@this\b'),
        re.compile(r'@parameter\d+\b'),
        re.compile(r'\bvirtualinvoke\b'),
        re.compile(r'\bstaticinvoke\b'),
        re.compile(r'\bspecialinvoke\b'),
        re.compile(r'\bgoto\b'),
        re.compile(r'\bLABEL\d+\b'),
        re.compile(r'class\s+"L[^"]+";'),
        re.compile(r'<[^>]+:\s[^>]+>'),
    ]

    def looks_like_java(text: str) -> bool:
        """Kiểm tra xem nội dung có phải Java code thực sự không."""
        t = text.strip()
        if not t:
            return False
        # Nếu chứa đặc trưng IR → loại ngay
        for pat in NON_JAVA_IR_PATTERNS:
            if pat.search(t):
                return False
        # Có class/method Java → ok
        if JAVA_CLASS_DECL.search(t) or JAVA_METHOD_SIG.search(t):
            return True
        # Nếu ≥2 keyword Java và có { hoặc ; → khả năng là Java
        kw_count = sum(1 for kw in JAVA_KEYWORDS if kw in t)
        if kw_count >= 2 and ("{" in t or ";" in t):
            return True
        return False

    # --- Duyệt thư mục ---
    deleted, checked = [], 0
    for name in java_files:
        checked += 1
        path = os.path.join(java_dir, name)
        try:
            with open(path, "r", encoding="utf-8", errors="ignore") as f:
                content = f.read()
            if not looks_like_java(content):
                os.remove(path)
                deleted.append(name)
        except Exception as e:
            print(f"⚠️ Lỗi khi xử lý {name}: {e}")

    # --- Báo cáo ---
    print(f"Checked {checked} .java files in '{java_dir}'.")
    print(f"Deleted {len(deleted)} non-Java files:")
    # for f in deleted:
    #     print(" -", f)

    return {"checked": checked, "deleted": deleted}

In [62]:
level_1_directory_arr = get_all_subdirectories_under_one_level(taint_result_root_directory)
# print(level_1_directory_arr)

In [63]:
for i, level_1_path in enumerate(level_1_directory_arr):
    print("Level_1 path:", level_1_path)
    package_name = get_app_package_name(level_1_path)
    print(f"==== Package {i}: {package_name} ====")
    vulnerability_directory = level_1_path+"\\vulnerability"
    # print(vulnerability_directory)
    output_dir = "java_blocks\\"+package_name
    html_path_arr = get_html_files_in_directory(vulnerability_directory)   
    for j in range(len(html_path_arr)):
        html_path = html_path_arr[j]
        extract_java_blocks_from_html(html_path, output_dir)
        # break
    cleanup_non_java_files(output_dir)
    # break

Level_1 path: C:\Users\ASUS\anaconda3-project-code\wearable-2-appshark\appshark-output\app-standalone\app.groupcal.www.apk
==== Package 0: app.groupcal.www.apk ====
Checked 187 .java files in 'java_blocks\app.groupcal.www.apk'.
Deleted 148 non-Java files:
Level_1 path: C:\Users\ASUS\anaconda3-project-code\wearable-2-appshark\appshark-output\app-standalone\apps.r.compass.apk
==== Package 1: apps.r.compass.apk ====
Checked 563 .java files in 'java_blocks\apps.r.compass.apk'.
Deleted 452 non-Java files:
Level_1 path: C:\Users\ASUS\anaconda3-project-code\wearable-2-appshark\appshark-output\app-standalone\ch.publisheria.bring.apk
==== Package 2: ch.publisheria.bring.apk ====
Checked 254 .java files in 'java_blocks\ch.publisheria.bring.apk'.
Deleted 196 non-Java files:
Level_1 path: C:\Users\ASUS\anaconda3-project-code\wearable-2-appshark\appshark-output\app-standalone\com.anghami.apk
==== Package 3: com.anghami.apk ====
Checked 7714 .java files in 'java_blocks\com.anghami.apk'.
Deleted 5796